# SAM3 Teacher Baseline — Feature Extraction & Tracking Benchmark

**Goal:** Establish the SAM3 teacher baseline for the pig-tracking pipeline: run SAM3 video tracking on the benchmark clips, measure tracking quality (MOTA / IDF1) and inference speed, and extract teacher features (backbone + 4-level FPN) and masks to disk for downstream knowledge distillation.

**Pipeline:**
1. Mount the data store and load the `facebook/sam3` model + processor
2. Stream each video frame-by-frame through SAM3 with bounding-box prompts
3. Record per-frame latency, throughput, GPU memory and MOTMetrics tracking quality
4. Extract and save teacher features (backbone, FPN), binary masks and bbox annotations

**Project roadmap (distillation):** feature distillation for SAM3 · standard KD for DINOv3 → smaller ViT · QAT post-distillation for both · (optional) explore SAM2-MobileViT variants.

In [0]:

# CELL 1: MOUNT BLOB STORAGE

try:
    dbutils.fs.unmount("/mnt/playbehavior")
except Exception as e:
    pass

storage_account_name = "<STORAGE_ACCOUNT>"
container_name = "<CONTAINER>"
storage_account_key = "<REDACTED_AZURE_STORAGE_ACCOUNT_KEY>"

dbutils.fs.mount(
    source=f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net",
    mount_point="/mnt/playbehavior",
    extra_configs={
        f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": storage_account_key
    }
)
print("Mounted /mnt/playbehavior")

/mnt/playbehavior has been unmounted.
Mounted /mnt/playbehavior


In [0]:

# CELL 2: SAFE INSTALLATION (PINS NUMPY < 2.0)

# 1. Install utilities, BUT forbid Numpy 2.0+ which crashes Databricks
# We use quotes to ensure the < operator is read correctly
%pip install "numpy<2.0" "opencv-python-headless" "motmetrics" "loguru" "accelerate"

# 2. Upgrade transformers safely
%pip install "transformers" --upgrade

print("✓ Installation complete.")
import numpy
print(f"✓ Numpy Version: {numpy.__version__} (Must be < 2.0)")
import cv2
print(f"✓ OpenCV Version: {cv2.__version__}")

INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/50.0 MB ? eta -:--:--
   ━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/50.0 MB 173.8 MB/s eta 0:00:01
   ━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/50.0 MB 201.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/50.0 MB 203.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━ 26.7/50.0 MB 214.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━ 34.0/50.0 MB 214.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━ 41.2/50.0 MB 209.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━ 48.5/50.0 MB 212.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 50.0/50.0 MB 209.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 50.0/50.0 MB 209.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 50.0/50.

In [0]:

# CELL 3: RESTART PYTHON

dbutils.library.restartPython()

In [0]:

# CELL 4: IMPORTS AND GPU SETUP

import os
import os.path as osp
import json
import time
import gc
import argparse
from collections import defaultdict
from datetime import datetime
from typing import Dict, List, Tuple, Optional

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.cuda.amp import autocast, GradScaler
from torchvision import transforms  # ← Make sure this is here
from PIL import Image              # ← Add this
import glob      

# Hugging Face SAM3 imports
try:
    from transformers import Sam3TrackerVideoModel, Sam3TrackerVideoProcessor
    from accelerate import Accelerator
    print("✓ Successfully imported SAM3 from Hugging Face transformers")
except ImportError as e:
    print(f"✗ Error importing SAM3: {e}")
    print("Make sure you have the latest transformers installed: pip install --upgrade transformers")

# Initialize accelerator for device management
accelerator = Accelerator()
device = accelerator.device

# Verify CUDA and GPU
print(f"\n{'='*70}")
print(f"GPU Configuration:")
print(f"{'='*70}")
print(f"Device: {device}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    gpu_props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory Total: {gpu_props.total_memory / 1024**3:.2f} GB")
    print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    print(f"GPU Memory Cached: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
    print(f"GPU Compute Capability: {gpu_props.major}.{gpu_props.minor}")
    # Check if it's an A10
    if "A10" in torch.cuda.get_device_name(0):
        print(f"✓ Detected NVIDIA A10 GPU - optimal for SAM3 benchmarking")
print(f"{'='*70}\n")

# Set default dtype for inference
DEFAULT_DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print(f"Default dtype for inference: {DEFAULT_DTYPE}")

✓ Successfully imported SAM3 from Hugging Face transformers

GPU Configuration:
Device: cuda:0
CUDA Available: True
GPU Device: NVIDIA A10-24Q
GPU Memory Total: 23.73 GB
GPU Memory Allocated: 0.00 GB
GPU Memory Cached: 0.00 GB
GPU Compute Capability: 8.6
✓ Detected NVIDIA A10 GPU - optimal for SAM3 benchmarking

Default dtype for inference: torch.bfloat16


In [0]:
from huggingface_hub import login

print("\nAuthenticating with HuggingFace...")

# METHOD 1: Direct token (replace with your token)
HF_TOKEN = "" 
login(token=HF_TOKEN)


Authenticating with HuggingFace...


In [0]:

# CELL 5: LOAD SAM3 MODEL AND PROCESSOR

print("Loading SAM3 model and processor from Hugging Face...")

# Load processor
processor = Sam3TrackerVideoProcessor.from_pretrained("facebook/sam3")
print("✓ Loaded SAM3 processor")

# Load model with bfloat16 for memory efficiency
model = Sam3TrackerVideoModel.from_pretrained(
    "facebook/sam3",
    torch_dtype=DEFAULT_DTYPE,
).to(device)

print(f"✓ Loaded SAM3 model to {device} with dtype {DEFAULT_DTYPE}")

# Print model size
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel Statistics:")
print(f"  Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")
print(f"  Trainable parameters: {trainable_params:,} ({trainable_params/1e6:.2f}M)")
print(f"  Model size (approx): {total_params * 2 / 1024**3:.2f} GB (bfloat16)")  # 2 bytes per param

# Set model to eval mode for inference
model.eval()
print(f"✓ Model set to evaluation mode")

# Clear GPU cache
gc.collect()
torch.cuda.empty_cache()
print(f"\nGPU Memory after model loading: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

Loading SAM3 model and processor from Hugging Face...


processor_config.json:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

✓ Loaded SAM3 processor


config.json:   0%|          | 0.00/25.8k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/845 [00:00<?, ?it/s]

✓ Loaded SAM3 model to cuda:0 with dtype torch.bfloat16

Model Statistics:
  Total parameters: 465,782,146 (465.78M)
  Trainable parameters: 465,782,146 (465.78M)
  Model size (approx): 0.87 GB (bfloat16)
✓ Model set to evaluation mode

GPU Memory after model loading: 0.92 GB


In [0]:

# CELL 6: HELPER FUNCTIONS FOR LOADING DATA (REVISED)


def load_frames_from_directory(frames_dir: str, max_frames: Optional[int] = None) -> List[np.ndarray]:
    """
    Load video frames from a directory of JPEG images.
    
    Args:
        frames_dir: Path to directory containing frame images
        max_frames: Maximum number of frames to load (None = load all)
        
    Returns:
        List of frames as numpy arrays (H, W, C) in RGB format
    """
    # Handle /dbfs prefix
    if not frames_dir.startswith("/dbfs"):
        frames_dir = osp.join("/dbfs", frames_dir.lstrip("/"))
    
    if not osp.exists(frames_dir):
        raise ValueError(f"Frames directory does not exist: {frames_dir}")
    
    # Get sorted list of frame files
    frame_files = sorted([
        osp.join(frames_dir, f) 
        for f in os.listdir(frames_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    
    if max_frames is not None:
        frame_files = frame_files[:max_frames]
    
    print(f"Loading {len(frame_files)} frames from {frames_dir}")
    
    # Load frames
    frames = []
    for frame_path in frame_files:
        # OpenCV loads in BGR, convert to RGB
        img = cv2.imread(frame_path)
        if img is None:
            print(f"Warning: Failed to load {frame_path}")
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        frames.append(img_rgb)
    
    print(f"✓ Loaded {len(frames)} frames, shape: {frames[0].shape if frames else 'N/A'}")
    return frames


def load_bounding_box_prompts(txt_path: str) -> Dict[str, List[int]]:
    """
    Load initial bounding box prompts from text file.
    Expected format: <object_name>: x,y,w,h
    
    Args:
        txt_path: Path to text file with bounding boxes
        
    Returns:
        Dictionary mapping object names to bounding boxes in [x1, y1, x2, y2] format
    """
    # Handle /dbfs prefix
    if not txt_path.startswith("/dbfs"):
        txt_path = osp.join("/dbfs", txt_path.lstrip("/"))
    
    if not osp.exists(txt_path):
        raise ValueError(f"Bounding box file does not exist: {txt_path}")
    
    prompts = {}
    with open(txt_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or ':' not in line:
                continue
            
            # Parse "object_name: x,y,w,h"
            name, coords = line.split(':', 1)
            name = name.strip()
            coords = coords.strip()
            
            # Parse coordinates
            x, y, w, h = map(float, coords.split(','))
            # Convert from [x, y, w, h] to [x1, y1, x2, y2] for SAM3
            bbox = [int(x), int(y), int(x + w), int(y + h)]
            prompts[name] = bbox
    
    print(f"✓ Loaded {len(prompts)} object prompts from {txt_path}")
    for name, bbox in prompts.items():
        print(f"  {name}: {bbox}")
    
    return prompts


def prepare_sam3_bbox_format(bboxes: List[List[int]]) -> List[List[List[int]]]:
    """
    Convert list of bboxes [x1, y1, x2, y2] to SAM3 input format.
    
    Args:
        bboxes: List of bounding boxes, each in [x1, y1, x2, y2] format
        
    Returns:
        Bboxes in SAM3 format: [[[x1, y1, x2, y2], [x1, y1, x2, y2], ...]]
        Shape: [1, num_objects, 4]
    """
    return [bboxes]  # Wrap in single batch dimension


def get_prompt_path_from_video_path(video_path: str) -> str:
    """
    Generate prompt file path from video path following the naming convention.
    
    Example:
        Input:  /mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_11_000036/video1
        Output: /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Initial Prompts/2019_11_11_000036/video1/BoundingBoxes_2019_11_11_000036.txt
    
    Args:
        video_path: Path to video frames directory
        
    Returns:
        Path to corresponding bounding box prompt file
    """
    # Extract video ID (e.g., "2019_11_11_000036")
    parts = video_path.rstrip('/').split('/')
    video_id = parts[-2]  # Second to last part
    video_name = parts[-1]  # "video1"
    
    # Construct prompt path
    prompt_path = f"/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Initial Prompts/{video_id}/{video_name}/BoundingBoxes_{video_id}.txt"
    
    return prompt_path


def get_output_path_from_video_path(video_path: str, base_dir: str = "/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Mask coordinates") -> str:
    """
    Generate output directory path from video path.
    
    Example:
        Input:  /mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_11_000036/video1
        Output: /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Mask coordinates/2019_11_11_000036
    
    Args:
        video_path: Path to video frames directory
        base_dir: Base directory for output
        
    Returns:
        Output directory path for this video
    """
    # Extract video ID
    parts = video_path.rstrip('/').split('/')
    video_id = parts[-2]
    
    output_path = osp.join(base_dir, video_id)
    os.makedirs(output_path, exist_ok=True)
    
    return output_path


def prepare_frames_or_path(video_path: str) -> str:
    """
    Validates that video_path exists and returns the correct path with /dbfs prefix.
    
    Args:
        video_path: Path to video file or frames directory
        
    Returns:
        Validated path with /dbfs prefix
    """
    if osp.exists(video_path):
        return video_path
    
    dbfs_path = osp.join("/dbfs", video_path.lstrip("/"))
    if osp.exists(dbfs_path):
        return dbfs_path
    
    raise ValueError(f"Path does not exist: {video_path}")

In [0]:
# CELL 5.5: BENCHMARK CONFIGURATION (UPDATED PATHS)
# ------------------------------------------------------------------------------

# List of all videos to process
BENCHMARK_VIDEOS = [
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_05_000002/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_11_000028/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_11_000036/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_22_000010/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_28_000113/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_12_02_000005/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_12_02_000208/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_12_10_000060/video1",
    "/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_12_10_000078/video1",
]

# Base directories
SAM3_OUTPUT_BASE = "/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Mask coordinates"
SAM3_METRICS_DIR = "/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Metrics"
SAM3_FEATURES_DIR = "/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Features"

# Create directories
os.makedirs(SAM3_OUTPUT_BASE, exist_ok=True)
os.makedirs(SAM3_METRICS_DIR, exist_ok=True)
os.makedirs(SAM3_FEATURES_DIR, exist_ok=True)

print(f"✓ Configuration loaded:")
print(f"  Videos:       {len(BENCHMARK_VIDEOS)}")
print(f"  Output base:  {SAM3_OUTPUT_BASE}")
print(f"  Metrics:      {SAM3_METRICS_DIR}")
print(f"  Features:     {SAM3_FEATURES_DIR}")

# Verify paths
print(f"\nVerifying video paths...")
for video_path in BENCHMARK_VIDEOS:
    full_path = prepare_frames_or_path(video_path)
    prompt_path = get_prompt_path_from_video_path(video_path)
    video_id = osp.basename(osp.dirname(video_path))
    print(f"  ✓ {video_id}: Frames={osp.exists(full_path)}, Prompts={osp.exists(prompt_path)}")

✓ Configuration loaded:
  Videos:       9
  Output base:  /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Mask coordinates
  Metrics:      /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Metrics
  Features:     /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Features

Verifying video paths...
  ✓ 2019_11_05_000002: Frames=True, Prompts=True
  ✓ 2019_11_11_000028: Frames=True, Prompts=True
  ✓ 2019_11_11_000036: Frames=True, Prompts=True
  ✓ 2019_11_22_000010: Frames=True, Prompts=True
  ✓ 2019_11_28_000113: Frames=True, Prompts=True
  ✓ 2019_12_02_000005: Frames=True, Prompts=True
  ✓ 2019_12_02_000208: Frames=True, Prompts=True
  ✓ 2019_12_10_000060: Frames=True, Prompts=True
  ✓ 2019_12_10_000078: Frames=True, Prompts=True


In [0]:
# CELL 7: COMPREHENSIVE PERFORMANCE TRACKER (FIXED)
# ------------------------------------------------------------------------------

class PerformanceTracker:
    """
    Comprehensive metrics tracking for SAM3 baseline and distilled models.
    
    FIXES vs previous version:
    - Per-frame inference time now correctly measured (wall clock per frame)
    - GPU peak memory now correctly captured via reset_peak_memory_stats()
    - Throughput stored as single float, not list
    - Two clean hooks: start_video_inference() / end_video_inference()
    """
    def __init__(
        self,
        model_name: str,
        model=None,
        log_dir: str = "/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Metrics"
    ):
        self.model_name = model_name
        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)

        # ── Per-frame timing ──────────────────────────────────────────────────
        self._frame_times_s = []        # Wall-clock seconds per frame
        self._video_start_time = None   # For overall FPS calculation
        self._frame_start_time = None   # For per-frame timing

        # ── GPU memory ───────────────────────────────────────────────────────
        self._gpu_memory_peak_gb = None     # Peak during entire video inference
        self._gpu_memory_before_gb = None   # Baseline before inference starts

        # ── Throughput ───────────────────────────────────────────────────────
        self._throughput_fps = None
        self.total_frames = 0

        # ── Mask quality ─────────────────────────────────────────────────────
        self._mask_areas = []

        # ── Timing breakdown (optional component profiling) ───────────────────
        self._component_timers = {}     # Active timers
        self._timing_breakdown = {}     # Recorded times: {component: [seconds]}

        # ── Tracking quality (from MOTMetrics) ───────────────────────────────
        self._tracking_quality = {}

        # ── Model size (computed once if model provided) ──────────────────────
        self.model_size_metrics = {}
        if model is not None:
            self.model_size_metrics = self._compute_model_size(model)

        # ── Legacy timer (for session init only) ─────────────────────────────
        self._legacy_timer_start = None

    # =========================================================================
    # MODEL SIZE
    # =========================================================================

    def _compute_model_size(self, model) -> dict:
        total = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        # SAM3 is loaded as bfloat16 → 2 bytes per param
        size_bytes = total * 2
        return {
            "total_parameters":             int(total),
            "total_parameters_millions":    round(total / 1e6, 6),
            "trainable_parameters":         int(trainable),
            "trainable_parameters_millions": round(trainable / 1e6, 6),
            "model_size_mb":                round(size_bytes / 1024**2, 4),
            "model_size_gb":                round(size_bytes / 1024**3, 6),
        }

    # =========================================================================
    # LEGACY TIMER  (used only for session init timing in process_single_video)
    # =========================================================================

    def start_timer(self):
        """Legacy: time a one-off event (e.g. session initialisation)."""
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self._legacy_timer_start = time.time()

    def end_timer(self) -> float:
        """Legacy: end one-off timer, return elapsed seconds."""
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.time() - self._legacy_timer_start
        self._legacy_timer_start = None
        return elapsed

    # =========================================================================
    # VIDEO-LEVEL HOOKS  ← The two new primary methods
    # =========================================================================

    def start_video_inference(self):
        """
        Call ONCE immediately before the frame loop starts.
        - Records GPU memory baseline
        - Resets peak memory counter
        - Starts wall-clock timer
        """
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            self._gpu_memory_before_gb = torch.cuda.memory_allocated() / 1024**3

        self._video_start_time = time.time()

    def end_video_inference(self, num_frames: int):
        """
        Call ONCE immediately after the frame loop ends.
        - Captures peak GPU memory over the whole video
        - Computes overall FPS
        
        Args:
            num_frames: Number of frames that were processed
        """
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            self._gpu_memory_peak_gb = torch.cuda.max_memory_allocated() / 1024**3

        total_time = time.time() - self._video_start_time
        self._throughput_fps = num_frames / total_time
        self.total_frames += num_frames

    # =========================================================================
    # PER-FRAME HOOKS
    # =========================================================================

    def start_frame(self):
        """Call immediately before each frame's inference."""
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self._frame_start_time = time.time()

    def end_frame(self):
        """Call immediately after each frame's inference."""
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        if self._frame_start_time is not None:
            self._frame_times_s.append(time.time() - self._frame_start_time)
            self._frame_start_time = None

    # =========================================================================
    # COMPONENT TIMING  (optional, for detailed breakdown)
    # =========================================================================

    def start_component_timer(self, component: str):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        self._component_timers[component] = time.time()

    def end_component_timer(self, component: str) -> float:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.time() - self._component_timers.pop(component, time.time())
        self._timing_breakdown.setdefault(component, []).append(elapsed)
        return elapsed

    # =========================================================================
    # MASK QUALITY
    # =========================================================================

    def record_mask_quality(self, mask_area: int = 0, contour_points: int = 0):
        if mask_area > 0:
            self._mask_areas.append(mask_area)

    # =========================================================================
    # TRACKING QUALITY
    # =========================================================================

    def add_tracking_quality_metrics(self, mot_summary: dict):
        """
        Accepts raw MOTMetrics output (values in 0-1 range for ratios).
        Stores percentages for display, raw values for JSON.
        """
        pct_keys = {'idf1', 'idp', 'idr', 'recall', 'precision', 'mota', 'motp'}
        for k, v in mot_summary.items():
            if k in pct_keys:
                self._tracking_quality[k] = float(v * 100)
            else:
                self._tracking_quality[k] = int(v) if isinstance(v, (int, float)) else v

    # =========================================================================
    # LEGACY COMPATIBILITY
    # =========================================================================

    def increment_frames(self, count: int = 1):
        """Legacy: kept for compatibility, end_video_inference handles totals."""
        pass

    def record_throughput(self, num_frames: int, elapsed_time: float):
        """Legacy: kept for compatibility, end_video_inference handles FPS."""
        pass

    def record_gpu_memory(self):
        """Legacy: kept for compatibility, end_video_inference handles GPU memory."""
        pass

    # =========================================================================
    # SUMMARY + SAVE
    # =========================================================================

    def get_summary(self) -> dict:
        fps = self._throughput_fps or 0.0

        # Per-frame stats (ms)
        if self._frame_times_s:
            mean_ms  = float(np.mean(self._frame_times_s)  * 1000)
            std_ms   = float(np.std(self._frame_times_s)   * 1000)
            min_ms   = float(np.min(self._frame_times_s)   * 1000)
            max_ms   = float(np.max(self._frame_times_s)   * 1000)
        else:
            # Derive from FPS if per-frame hooks weren't used
            mean_ms = (1000.0 / fps) if fps > 0 else 0.0
            std_ms = min_ms = max_ms = 0.0

        # Timing breakdown
        breakdown = {}
        for comp, times in self._timing_breakdown.items():
            breakdown[f"{comp}_mean_ms"]   = float(np.mean(times) * 1000)
            breakdown[f"{comp}_std_ms"]    = float(np.std(times)  * 1000)
            pct = (np.mean(times) / (mean_ms / 1000) * 100) if mean_ms > 0 else 0
            breakdown[f"{comp}_pct"]       = float(pct)

        return {
            "model_name":   self.model_name,
            "total_frames": self.total_frames,

            "model_size_metrics": self.model_size_metrics,

            "performance_metrics": {
                # Per-frame inference time (correctly measured)
                "inference_time_ms_mean":   mean_ms,
                "inference_time_ms_std":    std_ms,
                "inference_time_ms_min":    min_ms,
                "inference_time_ms_max":    max_ms,
                # Overall throughput
                "throughput_fps":           round(fps, 4),
                "latency_ms_per_frame":     round(1000.0 / fps, 2) if fps > 0 else None,
                # GPU memory (correctly measured)
                "gpu_memory_peak_gb":       round(self._gpu_memory_peak_gb, 4)
                                            if self._gpu_memory_peak_gb is not None else None,
                "gpu_memory_before_gb":     round(self._gpu_memory_before_gb, 4)
                                            if self._gpu_memory_before_gb is not None else None,
            },

            "mask_quality_metrics": {
                "mask_areas_mean": float(np.mean(self._mask_areas)) if self._mask_areas else 0.0,
                "mask_areas_std":  float(np.std(self._mask_areas))  if self._mask_areas else 0.0,
            },

            "timing_breakdown_metrics": breakdown,

            "tracking_quality_metrics": self._tracking_quality,
        }

    def save_metrics(self, video_id: str, suffix: str = "") -> str:
        filename = f"{self.model_name}_{video_id}_metrics{suffix}.json"
        filepath = osp.join(self.log_dir, filename)

        with open(filepath, "w") as f:
            json.dump({
                "video_id":   video_id,
                "model_name": self.model_name,
                "timestamp":  datetime.now().strftime("%Y%m%d_%H%M%S"),
                "summary":    self.get_summary(),
            }, f, indent=2)

        print(f"✓ Metrics saved to: {filepath}")
        return filepath

    def print_summary(self):
        s   = self.get_summary()
        pm  = s["performance_metrics"]
        mm  = s.get("model_size_metrics", {})
        mq  = s["mask_quality_metrics"]
        bd  = s["timing_breakdown_metrics"]
        tq  = s["tracking_quality_metrics"]

        print(f"\n{'='*70}")
        print(f"Comprehensive Metrics Summary: {self.model_name}")
        print(f"{'='*70}")

        if mm:
            print(f"\n【Model Size】")
            print(f"  Total Parameters:  {mm['total_parameters_millions']:.2f}M")
            print(f"  Model Size:        {mm['model_size_mb']:.2f} MB ({mm['model_size_gb']:.3f} GB)")

        print(f"\n【Performance】")
        print(f"  Inference Time:    {pm['inference_time_ms_mean']:.1f} ± {pm['inference_time_ms_std']:.1f} ms/frame")
        print(f"  Latency:           {pm['latency_ms_per_frame']} ms/frame")
        print(f"  Throughput:        {pm['throughput_fps']:.2f} FPS")
        gpu = pm['gpu_memory_peak_gb']
        print(f"  GPU Memory Peak:   {f'{gpu:.3f} GB' if gpu is not None else 'N/A'}")
        gpu_before = pm['gpu_memory_before_gb']
        print(f"  GPU Memory Before: {f'{gpu_before:.3f} GB' if gpu_before is not None else 'N/A'}")

        if bd:
            print(f"\n【Latency Breakdown】")
            for comp in ["preprocessing", "encoder", "decoder",
                         "propagation", "postprocessing"]:
                if f"{comp}_mean_ms" in bd:
                    print(f"  {comp.capitalize():<16} "
                          f"{bd[f'{comp}_mean_ms']:6.1f} ms  "
                          f"({bd[f'{comp}_pct']:5.1f}%)")

        if mq["mask_areas_mean"] > 0:
            print(f"\n【Mask Quality】")
            print(f"  Avg Mask Area:     {mq['mask_areas_mean']:.1f} ± {mq['mask_areas_std']:.1f} pixels")

        if tq:
            print(f"\n【Tracking Quality】")
            print(f"  MOTA:              {tq.get('mota', 0):.1f}%")
            print(f"  IDF1:              {tq.get('idf1', 0):.1f}%")
            print(f"  Precision:         {tq.get('precision', 0):.1f}%")
            print(f"  Recall:            {tq.get('recall', 0):.1f}%")
            print(f"  ID Switches:       {tq.get('num_switches', 0)}")

        print(f"{'='*70}\n")


print("✓ PerformanceTracker (fixed) defined")

✓ PerformanceTracker (fixed) defined


In [0]:

# CELL 8: TRACKING QUALITY EVALUATION FUNCTION

import motmetrics as mm

def evaluate_tracking_quality(video_id: str, 
                              track_dir: str, 
                              iou_threshold: float = 0.5) -> dict:
    """
    Evaluate tracking quality using MOTMetrics.
    
    Args:
        video_id: Video identifier (e.g., "2019_11_05_000002")
        track_dir: Directory containing tracking results (annotations_id*.json files)
        iou_threshold: IoU threshold for matching (default: 0.5)
        
    Returns:
        Dictionary of tracking quality metrics
    """
    # Construct ground truth path
    # Example: "2019_11_05_000002" -> "2019_11_05" and "000002"
    date_part = video_id[:10]  # "2019_11_05"
    seq_part = video_id[-6:]   # "000002"
    
    # Ground truth path format: /dbfs/.../annotated/2019_11_05/000002/output.json
    gt_json = f"/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/annotated/{date_part}/{seq_part}/output.json"
    
    if not osp.exists(gt_json):
        print(f"⚠ Ground truth not found: {gt_json}")
        return {}
    
    # Load ground truth
    with open(gt_json, "r") as f:
        gt_data = json.load(f)
    
    gt_boxes = {}  # (frame, id) → (x1, y1, x2, y2)
    for obj in gt_data["objects"]:
        pid = int(obj["id"])
        frames = obj["frames"]
        for i, fr in enumerate(frames):
            start = fr["frameNumber"]
            end = frames[i+1]["frameNumber"] if i+1 < len(frames) else 600
            x, y, w, h = fr["bbox"].values()
            box = (x, y, x + w, y + h)
            for f in range(start, end):
                gt_boxes[(f, pid)] = box
    
    # Load tracker results
    tr_boxes = {}  # (frame, id) → (x1, y1, x2, y2)
    for fname in os.listdir(track_dir):
        if not fname.startswith("annotations_id") or not fname.endswith(".json"):
            continue
        pid = int(fname.split()[-1].split(".")[0])
        data = json.load(open(os.path.join(track_dir, fname), "r"))
        for frame_name, ann in data.items():
            f = int(frame_name.split(".")[0]) - 1
            x, y, w, h = ann["bounding_box"]
            tr_boxes[(f, pid)] = (x, y, x + w, y + h)
    
    # Set up MOT accumulator
    acc = mm.MOTAccumulator(auto_id=True)
    
    # Feed in every frame
    for f in range(600):
        gt_ids = [pid for (ff, pid) in gt_boxes if ff == f]
        det_ids = [pid for (ff, pid) in tr_boxes if ff == f]
        
        cost = np.zeros((len(gt_ids), len(det_ids)), dtype=float)
        for i, gid in enumerate(gt_ids):
            gtb = gt_boxes[(f, gid)]
            for j, did in enumerate(det_ids):
                trb = tr_boxes[(f, did)]
                xa, ya = max(gtb[0], trb[0]), max(gtb[1], trb[1])
                xb, yb = min(gtb[2], trb[2]), min(gtb[3], trb[3])
                inter = max(0, xb - xa) * max(0, yb - ya)
                union = ((gtb[2] - gtb[0]) * (gtb[3] - gtb[1]) +
                        (trb[2] - trb[0]) * (trb[3] - trb[1]) -
                        inter)
                iou = inter / union if union > 0 else 0.0
                cost[i, j] = 1.0 - iou
        
        # Mask out matches worse than threshold
        cost[cost > (1.0 - iou_threshold)] = np.nan
        acc.update(gt_ids, det_ids, cost)
    
    # Compute metrics
    mh = mm.metrics.create()
    metrics = mm.metrics.motchallenge_metrics
    summary = mh.compute(acc, metrics=metrics, name=video_id)
    
    # Extract metrics as dictionary
    result = {}
    row = summary.loc[video_id]
    for name, val in row.items():
        result[name] = float(val) if isinstance(val, (float, np.floating)) else int(val)
    
    return result

In [0]:
# CELL 9: SAM3 STREAMING INFERENCE (FIXED - CORRECT PERFORMANCE TRACKING)
# ------------------------------------------------------------------------------
import torch
import os
import time
import gc
import json
import shutil
import numpy as np
import os.path as osp
from typing import Tuple, Optional
from PIL import Image

# --- FEATURE RECORDER ---
class OfflineFeatureRecorder:
    def __init__(self, model, output_base_dir):
        self.model = model
        self.final_output_dir = os.path.join(output_base_dir, "Features")
        self.local_temp_dir = "/local_disk0/tmp/sam3_features"
        self.hook_handle = None
        self.current_video_id = None
        self.frame_counter = 0
        
    def start_recording(self, video_id):
        self.current_video_id = video_id
        self.frame_counter = 0
        self.current_local_dir = os.path.join(self.local_temp_dir, video_id)
        if os.path.exists(self.current_local_dir):
            shutil.rmtree(self.current_local_dir)
        os.makedirs(self.current_local_dir, exist_ok=True)
        self.current_final_dir = os.path.join(self.final_output_dir, video_id)
        os.makedirs(self.current_final_dir, exist_ok=True)
        
        target_module = None
        if hasattr(self.model, "vision_encoder"):
            target_module = self.model.vision_encoder
            print(f"  ✓ Hooked: model.vision_encoder")
        elif hasattr(self.model, "sam_model"):
            if hasattr(self.model.sam_model, "vision_encoder"):
                target_module = self.model.sam_model.vision_encoder
                print(f"  ✓ Hooked: model.sam_model.vision_encoder")
            elif hasattr(self.model.sam_model, "image_encoder"):
                target_module = self.model.sam_model.image_encoder
                print(f"  ✓ Hooked: model.sam_model.image_encoder")
        elif hasattr(self.model, "image_encoder"):
            target_module = self.model.image_encoder
            print(f"  ✓ Hooked: model.image_encoder")
        
        if target_module is None:
            print("❌ ERROR: Could not find vision/image encoder.")
            return
        
        self.hook_handle = target_module.register_forward_hook(self._hook_fn)
        print(f"🔴 Recording to: {self.current_local_dir}")
    
    def _hook_fn(self, module, input, output):
        if not self.current_video_id:
            return
        try:
            if hasattr(output, 'last_hidden_state'):
                features = output.last_hidden_state.detach().cpu()
            elif hasattr(output, 'hidden_states') and output.hidden_states is not None:
                features = output.hidden_states[-1].detach().cpu()
            elif isinstance(output, tuple) and len(output) > 0:
                features = output[0].detach().cpu()
            elif isinstance(output, torch.Tensor):
                features = output.detach().cpu()
            else:
                return
            
            save_name = f"feat_{self.frame_counter:06d}.pt"
            save_path = os.path.join(self.current_local_dir, save_name)
            torch.save(features.clone().half(), save_path)
            self.frame_counter += 1
        except Exception as e:
            print(f"⚠️  Hook error at frame {self.frame_counter}: {e}")
    
    def stop_recording(self):
        if self.hook_handle:
            self.hook_handle.remove()
            self.hook_handle = None
        
        if self.current_video_id and os.path.exists(self.current_local_dir):
            print(f"📦 Transferring {self.frame_counter} features...")
            try:
                os.system(f"cp -r '{self.current_local_dir}/'* '{self.current_final_dir}/'")
                transferred = len([f for f in os.listdir(self.current_final_dir) if f.endswith('.pt')])
                
                if transferred == self.frame_counter:
                    print(f"  ✓ {transferred} features saved to DBFS")
                    shutil.rmtree(self.current_local_dir)
                else:
                    print(f"  ⚠️  Only {transferred}/{self.frame_counter} transferred")
            except Exception as e:
                print(f"  ❌ Transfer error: {e}")
            
            self.current_video_id = None
        print("🔴 Recording stopped")


# --- STREAMING PROCESSOR ---
def process_single_video_sam3(
    video_path: str,
    model,
    processor,
    tracker,                        # PerformanceTracker instance
    device: torch.device,
    output_base_dir: str,
    max_frames: Optional[int] = None,
    save_masks: bool = True,
    extract_features: bool = True,
    feature_save_base_dir: str = "/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/",
) -> Tuple[str, dict]:
    """Process video frame-by-frame using SAM3 streaming API."""
    
    video_id = osp.basename(osp.dirname(video_path))
    print(f"\n{'='*70}")
    print(f"Processing: {video_id} (Streaming)")
    print(f"{'='*70}")
    
    video_path_full = prepare_frames_or_path(video_path)
    prompt_path     = get_prompt_path_from_video_path(video_path)
    output_dir      = get_output_path_from_video_path(video_path, output_base_dir)
    
    print(f"  Frames:  {video_path_full}")
    print(f"  Prompts: {prompt_path}")
    print(f"  Output:  {output_dir}")
    
    # ── Collect frame paths ───────────────────────────────────────────────────
    frame_files = sorted([
        osp.join(video_path_full, f)
        for f in os.listdir(video_path_full)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    if max_frames is not None:
        frame_files = frame_files[:max_frames]
    print(f"  Frames to process: {len(frame_files)}")

    # ── Optional feature recorder ─────────────────────────────────────────────
    recorder = None
    if extract_features:
        print(f"  Feature extraction: ENABLED")
        recorder = OfflineFeatureRecorder(model, feature_save_base_dir)
        recorder.start_recording(video_id)

    # ── Load prompts ──────────────────────────────────────────────────────────
    print(f"\n[1/4] Loading prompts...")
    prompts      = load_bounding_box_prompts(prompt_path)
    object_names = sorted(prompts.keys())
    obj_ids_list = list(range(len(object_names)))
    bboxes       = [prompts[name] for name in object_names]
    input_boxes  = prepare_sam3_bbox_format(bboxes)
    print(f"  ✓ {len(object_names)} objects: {object_names}")

    # ── Init streaming session ────────────────────────────────────────────────
    print(f"\n[2/4] Initializing streaming session...")
    tracker.start_timer()                           # legacy: times session init only
    inference_session = processor.init_video_session(
        inference_device=device,
        dtype=DEFAULT_DTYPE,
    )
    init_time = tracker.end_timer()
    print(f"  ✓ Session ready ({init_time*1000:.1f} ms)")

    # ── Frame loop ────────────────────────────────────────────────────────────
    batch_annotations = {obj_id: {} for obj_id in obj_ids_list}

    print(f"\n[3/4] Processing frames...")

    # ▶▶▶ FIX 1: reset GPU peak counter + start wall clock BEFORE loop
    tracker.start_video_inference()

    total_start = time.time()   # kept for the print statement at end

    for frame_idx, frame_path in enumerate(frame_files):

        frame  = Image.open(frame_path).convert("RGB")
        inputs = processor(images=frame, device=device, return_tensors="pt")

        if frame_idx == 0:
            # CRITICAL: .copy() prevents processor mutating the list
            processor.add_inputs_to_inference_session(
                inference_session=inference_session,
                frame_idx=0,
                obj_ids=obj_ids_list.copy(),
                input_boxes=input_boxes,
                original_size=inputs.original_sizes[0],
            )
            print(f"  ✓ Prompts added to frame 0")

        # ▶▶▶ FIX 2: per-frame timer starts HERE (after preprocessing)
        tracker.start_frame()

        with torch.no_grad():
            sam_output = model(
                inference_session=inference_session,
                frame=inputs.pixel_values[0]
            )

        # ▶▶▶ FIX 3: per-frame timer ends HERE (pure model inference only)
        tracker.end_frame()

        # ── Post-process masks ────────────────────────────────────────────────
        video_res_masks = processor.post_process_masks(
            [sam_output.pred_masks],
            original_sizes=inputs.original_sizes,
            binarize=False
        )[0]

        for i, obj_id in enumerate(obj_ids_list):
            if len(video_res_masks.shape) == 4:
                mask_logits = video_res_masks[i, -1]
            elif len(video_res_masks.shape) == 3:
                mask_logits = video_res_masks[i]
            else:
                mask_logits = video_res_masks

            # Convert logits → probabilities → binary mask
            mask_probs  = torch.sigmoid(mask_logits)
            mask_binary = (mask_probs > 0.5).cpu().numpy()

            nonzero_pixels = np.argwhere(mask_binary)
            if len(nonzero_pixels) == 0:
                bbox      = [0, 0, 0, 0]
                mask_area = 0
            else:
                y_min, x_min = nonzero_pixels.min(axis=0).tolist()
                y_max, x_max = nonzero_pixels.max(axis=0).tolist()
                bbox      = [x_min, y_min, x_max - x_min, y_max - y_min]
                mask_area = int(mask_binary.sum())

            tracker.record_mask_quality(mask_area=mask_area)

            # Use actual filename (7-digit format)
            actual_frame_name = osp.basename(frame_path)
            batch_annotations[obj_id][actual_frame_name] = {
                "bounding_box": bbox,
                "object_name":  object_names[obj_id],
            }

        if (frame_idx + 1) % 50 == 0:
            print(f"  [{frame_idx + 1}/{len(frame_files)}] frames processed")

        del inputs, frame, sam_output, video_res_masks

        if (frame_idx + 1) % 100 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    # ▶▶▶ FIX 4: capture GPU peak + compute FPS AFTER loop
    tracker.end_video_inference(len(frame_files))

    if recorder:
        recorder.stop_recording()

    total_time = time.time() - total_start
    fps        = len(frame_files) / total_time
    print(f"\n  ✓ Complete: {len(frame_files)} frames in {total_time:.1f}s ({fps:.1f} FPS)")

    # ── Save annotations ──────────────────────────────────────────────────────
    if save_masks:
        print(f"\n[4/4] Saving annotations...")
        for obj_id, annotations in batch_annotations.items():
            ann_path = osp.join(output_dir, f"annotations_id {obj_id}.json")
            with open(ann_path, "w") as f:
                json.dump(annotations, f, indent=2)
        print(f"  ✓ {len(batch_annotations)} annotation files saved")

    del inference_session
    gc.collect()
    torch.cuda.empty_cache()

    return output_dir, {
        "video_id":      video_id,
        "num_frames":    len(frame_files),
        "num_objects":   len(obj_ids_list),
        "total_time":    total_time,
        "throughput_fps": fps,
    }


print("✓ Cell 9 ready (PerformanceTracker fully integrated)")

✓ Cell 9 ready (PerformanceTracker fully integrated)


In [0]:

# CELL 10: SAM3 BASELINE - BATCH PROCESSING ALL VIDEOS


def run_sam3_baseline_benchmark(
    video_paths: List[str],
    model: Sam3TrackerVideoModel,
    processor: Sam3TrackerVideoProcessor,
    device: torch.device,
    output_base_dir: str,
    metrics_dir: str,
    max_frames: Optional[int] = None,
    run_evaluation: bool = True,
) -> dict:
    """
    Run SAM3 baseline on all benchmark videos.
    
    Args:
        video_paths: List of video frame directories
        model: SAM3 model
        processor: SAM3 processor
        device: Device for inference
        output_base_dir: Base directory for outputs
        metrics_dir: Directory to save metrics
        max_frames: Max frames per video (None = all)
        run_evaluation: Whether to run tracking evaluation
        
    Returns:
        Dictionary of results for all videos
    """
    print(f"\n{'#'*70}")
    print(f"SAM3 BASELINE BENCHMARK - BATCH PROCESSING")
    print(f"{'#'*70}")
    print(f"Total videos: {len(video_paths)}")
    print(f"Output base:  {output_base_dir}")
    print(f"Metrics dir:  {metrics_dir}")
    print(f"{'#'*70}\n")
    
    # Initialize global tracker for model size metrics
    global_tracker = PerformanceTracker(
        model_name="SAM3_baseline",
        model=model,
        log_dir=metrics_dir
    )
    
    # Results storage
    all_results = {}
    
    # Process each video
    for video_idx, video_path in enumerate(video_paths):
        video_id = osp.basename(osp.dirname(video_path))
        
        print(f"\n{'='*70}")
        print(f"Video {video_idx + 1}/{len(video_paths)}: {video_id}")
        print(f"{'='*70}")
        
        # Create per-video tracker
        video_tracker = PerformanceTracker(
            model_name="SAM3_baseline",
            model=model,
            log_dir=metrics_dir
        )
        
        try:
            # Process video
            output_dir, video_info = process_single_video_sam3(
                video_path=video_path,
                model=model,
                processor=processor,
                tracker=video_tracker,
                device=device,
                output_base_dir=output_base_dir,
                max_frames=max_frames,
                save_masks=True,
            )
            
            # Run evaluation if requested
            if run_evaluation:
                print(f"\n[Evaluation] Running MOTMetrics evaluation...")
                try:
                    # Get video subdirectory name (e.g., "video1")
                    video_subdir = osp.basename(video_path)
                    track_dir = osp.join(output_dir, video_subdir)
                    
                    # Create track_dir if it doesn't exist yet
                    if not osp.exists(track_dir):
                        # Annotations are saved directly in output_dir
                        track_dir = output_dir
                    
                    tracking_metrics = evaluate_tracking_quality(
                        video_id=video_id,
                        track_dir=track_dir,
                        iou_threshold=0.5
                    )
                    
                    if tracking_metrics:
                        video_tracker.add_tracking_quality_metrics(tracking_metrics)
                        print(f"  ✓ Evaluation complete: MOTA={tracking_metrics.get('mota', 0)*100:.1f}%, "
                              f"IDF1={tracking_metrics.get('idf1', 0)*100:.1f}%")
                    else:
                        print(f"  ⚠ Evaluation skipped (no ground truth)")
                        
                except Exception as e:
                    print(f"  ✗ Evaluation failed: {e}")
            
            # Print summary for this video
            video_tracker.print_summary()
            
            # Save metrics for this video
            metrics_path = video_tracker.save_metrics(video_id=video_id)
            
            # Store results
            all_results[video_id] = {
                "status": "success",
                "output_dir": output_dir,
                "metrics_path": metrics_path,
                "summary": video_tracker.get_summary(),
            }
            
        except Exception as e:
            print(f"\n✗ Error processing {video_id}: {e}")
            import traceback
            traceback.print_exc()
            
            all_results[video_id] = {
                "status": "failed",
                "error": str(e),
            }
        
        # Clear GPU memory between videos
        gc.collect()
        torch.cuda.empty_cache()
        print(f"\nGPU Memory cleared: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB allocated")
    
    # Save aggregated results
    print(f"\n{'#'*70}")
    print(f"BATCH PROCESSING COMPLETE")
    print(f"{'#'*70}")
    
    summary_path = osp.join(metrics_dir, "SAM3_baseline_batch_summary.json")
    with open(summary_path, "w") as f:
        json.dump(all_results, f, indent=2)
    
    print(f"\n✓ Batch summary saved to: {summary_path}")
    
    # Print final statistics
    successful = sum(1 for r in all_results.values() if r["status"] == "success")
    failed = len(all_results) - successful
    
    print(f"\nFinal Statistics:")
    print(f"  Total videos:     {len(video_paths)}")
    print(f"  Successful:       {successful}")
    print(f"  Failed:           {failed}")
    
    if successful > 0:
        # Aggregate metrics across all successful videos
        avg_mota = np.mean([
            r["summary"]["tracking_quality_metrics"].get("mota", 0)
            for r in all_results.values()
            if r["status"] == "success" and r["summary"]["tracking_quality_metrics"]
        ])
        
        avg_fps = np.mean([
            r["summary"]["performance_metrics"].get("throughput_fps_mean", 0)
            for r in all_results.values()
            if r["status"] == "success"
        ])
        
        avg_gpu = np.mean([
            r["summary"]["performance_metrics"].get("gpu_memory_peak_mean", 0)
            for r in all_results.values()
            if r["status"] == "success"
        ])
        
        print(f"\nAggregated Metrics (across all videos):")
        print(f"  Avg MOTA:         {avg_mota:.1f}%")
        print(f"  Avg Throughput:   {avg_fps:.2f} FPS")
        print(f"  Avg GPU Memory:   {avg_gpu:.2f} GB")
    
    return all_results

In [0]:

# CELL 10.5: JETSON ORIN SIMULATION MODE (OPTIONAL)


SIMULATE_JETSON_ORIN = False  # Set to True to enable simulation

if SIMULATE_JETSON_ORIN:
    print("\n" + "="*70)
    print("JETSON ORIN SIMULATION MODE ENABLED")
    print("="*70)
    
    # Limit memory to simulate Jetson Orin's memory constraints
    # Orin has 32GB total, but ~8GB reserved for system
    # Available for models: ~20GB (conservative estimate)
    memory_fraction = 20 / 24  # 20GB out of 24GB available on A10
    torch.cuda.set_per_process_memory_fraction(memory_fraction, device=0)
    
    print(f"✓ Memory limited to {memory_fraction*100:.0f}% ({20:.0f} GB)")
    print(f"  This simulates Jetson AGX Orin 32GB memory constraints")
    print(f"  Note: Compute throughput on A10 will still be higher than Orin")
    print("="*70 + "\n")
else:
    print("\nJetson Orin simulation: DISABLED (using full A10 resources)")


Jetson Orin simulation: DISABLED (using full A10 resources)


In [0]:
# ------------------------------------------------------------------------------
# CELL 11 SIMPLIFIED: SINGLE PASS
# ------------------------------------------------------------------------------

def run_sam3_single_pass_corrected(
    video_paths: List[str],
    model,
    processor,
    device: torch.device,
    base_output_dir: str,
    metrics_dir: str,
    max_frames: Optional[int] = None,
    run_evaluation: bool = True,
) -> dict:
    """
    Single pass: Generate annotations + benchmark metrics.
    Features already exist from earlier run.
    """
    
    print(f"\n{'#'*70}")
    print(f"SAM3 SINGLE-PASS BENCHMARK (CORRECTED)")
    print(f"{'#'*70}")
    print(f"Total videos: {len(video_paths)}")
    print(f"Purpose: Generate annotations + measure baseline performance")
    print(f"Note: Features already extracted (skipping)")
    print(f"{'#'*70}\n")
    
    output_dir = osp.join(base_output_dir, "Corrected_Annotations")
    os.makedirs(output_dir, exist_ok=True)
    
    all_results = {}
    
    for video_idx, video_path in enumerate(video_paths):
        video_id = osp.basename(osp.dirname(video_path))
        
        print(f"\n[Video {video_idx + 1}/{len(video_paths)}]: {video_id}")
        print(f"{'='*70}")
        
        video_tracker = PerformanceTracker(
            model_name="SAM3_baseline",
            model=model if video_idx == 0 else None,
            log_dir=metrics_dir
        )
        
        try:
            video_output_dir, video_info = process_single_video_sam3(
                video_path=video_path,
                model=model,
                processor=processor,
                tracker=video_tracker,
                device=device,
                output_base_dir=output_dir,
                max_frames=max_frames,
                save_masks=True,  # ← Save annotations
                extract_features=False,  # ← Skip (already have)
                feature_save_base_dir="/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/",
            )
            
            # Evaluate
            if run_evaluation:
                print(f"\n[Evaluation] Running MOTMetrics...")
                try:
                    tracking_metrics = evaluate_tracking_quality(
                        video_id=video_id,
                        track_dir=video_output_dir,
                        iou_threshold=0.5
                    )
                    
                    if tracking_metrics:
                        video_tracker.add_tracking_quality_metrics(tracking_metrics)
                        print(f"  ✓ MOTA={tracking_metrics.get('mota', 0)*100:.1f}%, "
                              f"IDF1={tracking_metrics.get('idf1', 0)*100:.1f}%")
                    else:
                        print(f"  ⚠️ No ground truth available")
                        
                except Exception as e:
                    print(f"  ✗ Evaluation failed: {e}")
            
            video_tracker.print_summary()
            metrics_path = video_tracker.save_metrics(video_id=video_id)
            
            all_results[video_id] = {
                "status": "success",
                "output_dir": video_output_dir,
                "metrics_path": metrics_path,
                "summary": video_tracker.get_summary(),
            }
            
        except Exception as e:
            print(f"\n✗ Error: {e}")
            import traceback
            traceback.print_exc()
            all_results[video_id] = {"status": "failed", "error": str(e)}
        
        gc.collect()
        torch.cuda.empty_cache()
    
    # Summary
    print(f"\n{'#'*70}")
    print(f"BENCHMARK COMPLETE")
    print(f"{'#'*70}")
    
    summary_path = osp.join(metrics_dir, "SAM3_baseline_summary.json")
    with open(summary_path, "w") as f:
        json.dump(all_results, f, indent=2)
    
    successful = sum(1 for r in all_results.values() if r["status"] == "success")
    
    if successful > 0:
        avg_fps = np.mean([r["summary"]["performance_metrics"].get("throughput_fps_mean", 0) for r in all_results.values() if r["status"] == "success"])
        avg_mota = np.mean([r["summary"]["tracking_quality_metrics"].get("mota", 0) for r in all_results.values() if r["status"] == "success" and r["summary"]["tracking_quality_metrics"]])
        
        print(f"\n{'='*70}")
        print(f"SAM3 TEACHER BASELINE")
        print(f"{'='*70}")
        print(f"  Throughput:  {avg_fps:.2f} FPS")
        print(f"  MOTA:        {avg_mota:.1f}%")
        print(f"  Successful:  {successful}/{len(video_paths)} videos")
        print(f"{'='*70}")
        print(f"\n✓ Ready for distillation training!")
    
    return all_results


# Run it
results = run_sam3_single_pass_corrected(
    video_paths=BENCHMARK_VIDEOS,
    model=model,
    processor=processor,
    device=device,
    base_output_dir=SAM3_OUTPUT_BASE,
    metrics_dir=SAM3_METRICS_DIR,
    max_frames=None,
    run_evaluation=True,
)


######################################################################
SAM3 SINGLE-PASS BENCHMARK (CORRECTED)
######################################################################
Total videos: 9
Purpose: Generate annotations + measure baseline performance
Note: Features already extracted (skipping)
######################################################################


[Video 1/9]: 2019_11_05_000002

Processing: 2019_11_05_000002 (Streaming)
  Frames:  /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_05_000002/video1
  Prompts: /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Initial Prompts/2019_11_05_000002/video1/BoundingBoxes_2019_11_05_000002.txt
  Output:  /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Mask coordinates/Corrected_Annotations/2019_11_05_000002
  Frames to process: 600

[1/4] Loading prompts...
✓ Loaded 8 object prompts from /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Initial Prompts/2019_11_05_000002/video1/BoundingBoxes_201

## NEW-EXTRACTION

In [0]:
import shutil

base = "/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3"

# Old neck features (single-level, being replaced by 4-level FPN)
neck_dir = os.path.join(base, "Neck_Features")
if os.path.exists(neck_dir):
    shutil.rmtree(neck_dir)
    print(f"✓ Deleted: Neck_Features/")

# Old backbone features (not needed — FPN features replace these)
feat_dir = os.path.join(base, "Features")
if os.path.exists(feat_dir):
    shutil.rmtree(feat_dir)
    print(f"✓ Deleted: Features/")

# Old checkpoints (trained on wrong targets)
for ckpt_dir in ["Checkpoints", "Checkpoints_NeckDistill"]:
    d = os.path.join(base, ckpt_dir)
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"✓ Deleted: {ckpt_dir}/")

In [0]:

# CELL NEW-EXTRACTION: SINGLE-PASS SAM3 EXTRACTION (DBFS-SAFE)
# Saves: annotations (bbox JSON), binary masks (png), fpn features (pt)
# Uses local disk as buffer to avoid DBFS seek errors

import torch
import os
import os.path as osp
import time
import gc
import json
import shutil
import numpy as np
from PIL import Image
from typing import Optional, Tuple, List

# Output directories
BASE_OUTPUT = "/dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3"
ANNOT_OUTPUT = osp.join(BASE_OUTPUT, "Mask coordinates", "Teacher_Annotations")
MASK_OUTPUT = osp.join(BASE_OUTPUT, "Masks")
FPN_OUTPUT = osp.join(BASE_OUTPUT, "FPN_Features")
METRICS_OUTPUT = osp.join(BASE_OUTPUT, "Metrics")

# Local temp directory (fast SSD on the Databricks worker)
LOCAL_TEMP = "/local_disk0/tmp/sam3_extraction"

for d in [ANNOT_OUTPUT, MASK_OUTPUT, FPN_OUTPUT, METRICS_OUTPUT]:
    os.makedirs(d, exist_ok=True)


class FPNFeatureHook:
    """Captures all 4 FPN levels + position encodings from the vision encoder."""
    
    def __init__(self, model):
        self.captured_fpn = None
        self.captured_pos = None
        self.handle = None
        
        target = model.vision_encoder
        self.handle = target.register_forward_hook(self._hook_fn)
        print(f"  ✓ FPN hook attached to vision_encoder")
    
    def _hook_fn(self, module, input, output):
        if hasattr(output, 'fpn_hidden_states') and output.fpn_hidden_states is not None:
            self.captured_fpn = [x.detach().cpu().half() for x in output.fpn_hidden_states]
            self.captured_pos = [x.detach().cpu().half() for x in output.fpn_position_encoding]
    
    def get_and_clear(self):
        fpn, pos = self.captured_fpn, self.captured_pos
        self.captured_fpn = None
        self.captured_pos = None
        return fpn, pos
    
    def remove(self):
        if self.handle:
            self.handle.remove()
            self.handle = None


def save_mask_to_local_then_copy(mask_binary, local_path, dbfs_path):
    """
    Save binary mask as PNG via local disk to avoid DBFS seek errors.
    PNG is lossless for binary data and doesn't need seek.
    """
    # Save to local disk first
    os.makedirs(osp.dirname(local_path), exist_ok=True)
    
    # Save as single-channel PNG (lossless, no seek needed)
    mask_img = Image.fromarray(mask_binary * 255)  # 0/255 for visibility
    mask_img.save(local_path, format='PNG')


def save_tensor_to_local(tensor, local_path):
    """Save tensor to local disk."""
    os.makedirs(osp.dirname(local_path), exist_ok=True)
    torch.save(tensor, local_path)


def bulk_copy_to_dbfs(local_dir, dbfs_dir):
    """Copy entire directory from local disk to DBFS."""
    os.makedirs(dbfs_dir, exist_ok=True)
    
    # Use cp -r for speed
    ret = os.system(f"cp -r '{local_dir}/'* '{dbfs_dir}/' 2>/dev/null")
    
    if ret != 0:
        # Fallback: file-by-file copy
        for root, dirs, files in os.walk(local_dir):
            rel_root = osp.relpath(root, local_dir)
            dest_root = osp.join(dbfs_dir, rel_root)
            os.makedirs(dest_root, exist_ok=True)
            for f in files:
                shutil.copy2(osp.join(root, f), osp.join(dest_root, f))


def extract_single_video(
    video_path: str,
    model,
    processor,
    device: torch.device,
    annot_output_dir: str,
    mask_output_dir: str,
    FPN_OUTPUT_dir: str,
    metrics_dir: str,
    max_frames: Optional[int] = None,
) -> dict:
    """
    Single-pass extraction for one video.
    Writes to local SSD first, then bulk-copies to DBFS.
    """
    
    video_id = osp.basename(osp.dirname(video_path))
    frames_dir = prepare_frames_or_path(video_path)
    prompt_path = get_prompt_path_from_video_path(video_path)
    
    # LOCAL temp directories (fast writes)
    local_annot_dir = osp.join(LOCAL_TEMP, video_id, "annotations")
    local_mask_dir = osp.join(LOCAL_TEMP, video_id, "masks")
    local_fpn_dir = osp.join(LOCAL_TEMP, video_id, "fpn")
    
    # Clean any previous local temp
    local_video_dir = osp.join(LOCAL_TEMP, video_id)
    if osp.exists(local_video_dir):
        shutil.rmtree(local_video_dir)
    
    for d in [local_annot_dir, local_mask_dir, local_fpn_dir]:
        os.makedirs(d, exist_ok=True)
    
    # DBFS final directories
    dbfs_annot_dir = osp.join(annot_output_dir, video_id)
    dbfs_mask_dir = osp.join(mask_output_dir, video_id)
    dbfs_fpn_dir = osp.join(FPN_OUTPUT_dir, video_id)
    
    print(f"\n{'='*70}")
    print(f"Extracting: {video_id}")
    print(f"{'='*70}")
    print(f"  Frames:       {frames_dir}")
    print(f"  Local temp:   {local_video_dir}")
    print(f"  DBFS annot:   {dbfs_annot_dir}")
    print(f"  DBFS masks:   {dbfs_mask_dir}")
    print(f"  DBFS fpn:    {dbfs_fpn_dir}")
    
    # Collect frames
    frame_files = sorted([
        osp.join(frames_dir, f)
        for f in os.listdir(frames_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])
    if max_frames is not None:
        frame_files = frame_files[:max_frames]
    print(f"  Frames to process: {len(frame_files)}")
    
    # Load prompts
    prompts = load_bounding_box_prompts(prompt_path)
    object_names = sorted(prompts.keys())
    obj_ids_list = list(range(len(object_names)))
    bboxes = [prompts[name] for name in object_names]
    input_boxes = prepare_sam3_bbox_format(bboxes)
    print(f"  Objects ({len(object_names)}): {object_names}")
    
    # Attach fpn hook
    fpn_hook = FPNFeatureHook(model)
    
    # Init streaming session
    inference_session = processor.init_video_session(
        inference_device=device,
        dtype=DEFAULT_DTYPE,
    )
    
    # Storage for annotations
    batch_annotations = {obj_id: {} for obj_id in obj_ids_list}
    
    fpn_saved = 0
    mask_saved = 0
    
    start_time = time.time()
    
    for frame_idx, frame_path in enumerate(frame_files):
        
        actual_frame_name = osp.basename(frame_path)
        frame_stem = actual_frame_name.split('.')[0]
        
        frame = Image.open(frame_path).convert("RGB")
        inputs = processor(images=frame, device=device, return_tensors="pt")
        
        if frame_idx == 0:
            processor.add_inputs_to_inference_session(
                inference_session=inference_session,
                frame_idx=0,
                obj_ids=obj_ids_list.copy(),
                input_boxes=input_boxes,
                original_size=inputs.original_sizes[0],
            )
            print(f"  ✓ Prompts added to frame 0")
        
        # Forward pass
        with torch.no_grad():
            sam_output = model(
                inference_session=inference_session,
                frame=inputs.pixel_values[0]
            )

        fpn_feats, pos_encs = fpn_hook.get_and_clear()
        if fpn_feats is not None:
            fpn_path = osp.join(local_fpn_dir, f"fpn_{frame_idx:06d}.pt")
            torch.save({
                'fpn_hidden_states': fpn_feats,       # List of 4 tensors
                'fpn_position_encoding': pos_encs,     # List of 4 tensors
            }, fpn_path)
            fpn_saved += 1

        # ── Post-process masks ────────────────────────────────────────────
        video_res_masks = processor.post_process_masks(
            [sam_output.pred_masks],
            original_sizes=inputs.original_sizes,
            binarize=False
        )[0]
        
        for i, obj_id in enumerate(obj_ids_list):
            if len(video_res_masks.shape) == 4:
                mask_logits = video_res_masks[i, -1]
            elif len(video_res_masks.shape) == 3:
                mask_logits = video_res_masks[i]
            else:
                mask_logits = video_res_masks
            
            mask_probs = torch.sigmoid(mask_logits)
            mask_binary = (mask_probs > 0.5).cpu().numpy().astype(np.uint8)
            
            # ── Save binary mask as PNG (to local disk) ───────────────────
            obj_mask_dir = osp.join(local_mask_dir, object_names[obj_id])
            os.makedirs(obj_mask_dir, exist_ok=True)
            mask_img = Image.fromarray(mask_binary * 255)
            mask_img.save(osp.join(obj_mask_dir, f"{frame_stem}.png"), format='PNG')
            mask_saved += 1
            
            # ── Compute bbox from mask ────────────────────────────────────
            nonzero_pixels = np.argwhere(mask_binary)
            if len(nonzero_pixels) == 0:
                bbox = [0, 0, 0, 0]
                mask_area = 0
            else:
                y_min, x_min = nonzero_pixels.min(axis=0).tolist()
                y_max, x_max = nonzero_pixels.max(axis=0).tolist()
                bbox = [x_min, y_min, x_max - x_min, y_max - y_min]
                mask_area = int(mask_binary.sum())
            
            batch_annotations[obj_id][actual_frame_name] = {
                "bounding_box": bbox,
                "object_name": object_names[obj_id],
                "mask_area": mask_area,
            }
        
        if (frame_idx + 1) % 50 == 0:
            elapsed = time.time() - start_time
            fps = (frame_idx + 1) / elapsed
            print(f"  [{frame_idx + 1}/{len(frame_files)}] "
                  f"{fps:.1f} FPS | "
                  f"fpn: {fpn_saved} | Masks: {mask_saved}")
        
        del inputs, frame, sam_output, video_res_masks
        
        if (frame_idx + 1) % 100 == 0:
            gc.collect()
            torch.cuda.empty_cache()
    
    fpn_hook.remove()
    
    total_time = time.time() - start_time
    fps = len(frame_files) / total_time
    
    print(f"\n  ✓ Inference complete: {len(frame_files)} frames in {total_time:.1f}s ({fps:.1f} FPS)")
    print(f"    fpn features: {fpn_saved}")
    print(f"    Mask files:    {mask_saved}")
    
    # ── Save annotations to local disk ────────────────────────────────────
    for obj_id, annotations in batch_annotations.items():
        ann_path = osp.join(local_annot_dir, f"annotations_id {obj_id}.json")
        with open(ann_path, "w") as f:
            json.dump(annotations, f, indent=2)
    print(f"    Annotations:   {len(batch_annotations)} files")
    
    # ── Bulk copy from local → DBFS ──────────────────────────────────────
    print(f"\n  📦 Copying to DBFS...")
    
    copy_start = time.time()
    
    bulk_copy_to_dbfs(local_annot_dir, dbfs_annot_dir)
    dbfs_annot_count = len([f for f in os.listdir(dbfs_annot_dir) if f.endswith('.json')])
    print(f"    Annotations: {dbfs_annot_count} files → DBFS ✓")
    
    bulk_copy_to_dbfs(local_mask_dir, dbfs_mask_dir)
    # Count masks across all object subdirectories
    dbfs_mask_count = 0
    for obj_name in os.listdir(dbfs_mask_dir):
        obj_path = osp.join(dbfs_mask_dir, obj_name)
        if osp.isdir(obj_path):
            dbfs_mask_count += len([f for f in os.listdir(obj_path) if f.endswith('.png')])
    print(f"    Masks:       {dbfs_mask_count} files → DBFS ✓")
    
    bulk_copy_to_dbfs(local_fpn_dir, dbfs_fpn_dir)
    dbfs_fpn_count = len([f for f in os.listdir(dbfs_fpn_dir) if f.endswith('.pt')])
    print(f"    FPN:         {dbfs_fpn_count} files → DBFS ✓")
    
    copy_time = time.time() - copy_start
    print(f"    Copy time:   {copy_time:.1f}s")
    
    # ── Cleanup local temp ────────────────────────────────────────────────
    shutil.rmtree(local_video_dir)
    print(f"    Local temp cleaned ✓")
    
    del inference_session
    gc.collect()
    torch.cuda.empty_cache()
    
    return {
        "video_id": video_id,
        "num_frames": len(frame_files),
        "num_objects": len(object_names),
        "object_names": object_names,
        "FPN_Features_saved": dbfs_fpn_count,
        "masks_saved": dbfs_mask_count,
        "annotations_saved": dbfs_annot_count,
        "total_time_s": total_time,
        "fps": fps,
    }



# RUN EXTRACTION ON ALL VIDEOS


print("\n" + "#" * 70)
print("SAM3 FULL EXTRACTION: ANNOTATIONS + MASKS + fpn FEATURES")
print("#" * 70)
print(f"Videos:        {len(BENCHMARK_VIDEOS)}")
print(f"Annotations:   {ANNOT_OUTPUT}")
print(f"Masks:         {MASK_OUTPUT}")
print(f"fpn features: {FPN_OUTPUT}")
print(f"Local buffer:  {LOCAL_TEMP}")
print("#" * 70 + "\n")

all_extraction_results = {}

for video_idx, video_path in enumerate(BENCHMARK_VIDEOS):
    video_id = osp.basename(osp.dirname(video_path))
    
    print(f"\n[Video {video_idx + 1}/{len(BENCHMARK_VIDEOS)}]: {video_id}")
    
    # Check if already extracted
    video_fpn_dir = osp.join(FPN_OUTPUT, video_id)
    video_mask_dir = osp.join(MASK_OUTPUT, video_id)
    
    already_done = False
    if osp.exists(video_fpn_dir) and osp.exists(video_mask_dir):
        fpn_count = len([f for f in os.listdir(video_fpn_dir) if f.endswith('.pt')])
        # Check mask count across subdirs
        mask_count = 0
        if osp.isdir(video_mask_dir):
            for obj_dir in os.listdir(video_mask_dir):
                obj_path = osp.join(video_mask_dir, obj_dir)
                if osp.isdir(obj_path):
                    mask_count += len([f for f in os.listdir(obj_path) if f.endswith('.png')])
        
        if fpn_count >= 500 and mask_count >= 500:
            already_done = True
            print(f"  ✓ Already extracted (fpn: {fpn_count}, masks: {mask_count}), skipping")
            all_extraction_results[video_id] = {"status": "skipped"}
            continue
    
    try:
        result = extract_single_video(
            video_path=video_path,
            model=model,
            processor=processor,
            device=device,
            annot_output_dir=ANNOT_OUTPUT,
            mask_output_dir=MASK_OUTPUT,
            FPN_OUTPUT_dir=FPN_OUTPUT,
            metrics_dir=METRICS_OUTPUT,
            max_frames=None,
        )
        all_extraction_results[video_id] = {"status": "success", **result}
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        import traceback
        traceback.print_exc()
        all_extraction_results[video_id] = {"status": "failed", "error": str(e)}
    
    gc.collect()
    torch.cuda.empty_cache()


# VERIFICATION

print(f"\n{'#'*70}")
print("EXTRACTION COMPLETE — VERIFICATION")
print(f"{'#'*70}\n")

for video_path in BENCHMARK_VIDEOS:
    video_id = osp.basename(osp.dirname(video_path))
    
    # fpn
    fpn_dir = osp.join(FPN_OUTPUT, video_id)
    fpn_count = 0
    fpn_shape = "N/A"
    if osp.exists(fpn_dir):
        fpn_files = sorted([f for f in os.listdir(fpn_dir) if f.endswith('.pt')])
        fpn_count = len(fpn_files)
        if fpn_files:
            sample = torch.load(osp.join(fpn_dir, fpn_files[0]), map_location='cpu')
            fpn_shape = str([list(x.shape) for x in sample['fpn_hidden_states']])
    
    # Masks
    mask_dir = osp.join(MASK_OUTPUT, video_id)
    mask_count = 0
    mask_shape = "N/A"
    num_objects = 0
    if osp.exists(mask_dir):
        obj_dirs = [d for d in os.listdir(mask_dir) if osp.isdir(osp.join(mask_dir, d))]
        num_objects = len(obj_dirs)
        for obj_dir_name in obj_dirs:
            obj_path = osp.join(mask_dir, obj_dir_name)
            png_files = [f for f in os.listdir(obj_path) if f.endswith('.png')]
            mask_count += len(png_files)
            if png_files and mask_shape == "N/A":
                sample_mask = np.array(Image.open(osp.join(obj_path, png_files[0])))
                mask_shape = str(sample_mask.shape)
    
    # Annotations
    annot_dir = osp.join(ANNOT_OUTPUT, video_id)
    annot_count = 0
    if osp.exists(annot_dir):
        annot_count = len([f for f in os.listdir(annot_dir) if f.endswith('.json')])
    
    # Backbone features (existing)
    feat_dir = osp.join(BASE_OUTPUT, "Features", video_id)
    feat_count = 0
    if osp.exists(feat_dir):
        feat_count = len([f for f in os.listdir(feat_dir) if f.endswith('.pt')])
    
    status = "✓" if (fpn_count >= 500 and mask_count >= 500) else "⚠️"
    

    print(f"  {status} {video_id}:")
    print(f"      Backbone features: {feat_count:>4} files (existing)")
    print(f"      fpn features:     {fpn_count:>4} files, shape={fpn_shape}")
    print(f"      Masks:             {mask_count:>4} files across {num_objects} objects, shape={mask_shape}")
    print(f"      Annotations:       {annot_count:>4} files")

# Save summary
summary_path = osp.join(METRICS_OUTPUT, "extraction_summary.json")
with open(summary_path, "w") as f:
    json.dump(all_extraction_results, f, indent=2, default=str)
print(f"\n✓ Summary saved: {summary_path}")

print(f"\n{'='*70}")
print("DATA SPLIT REMINDER:")
print(f"{'='*70}")
print(f"  ALL 9 videos extracted (teacher ground truth)")
print(f"  Adapter training will use ONLY videos 0-3 (train) + 4 (val)")
print(f"  Videos 5-8 are HELD OUT for final evaluation")
print(f"  This matches the encoder distillation split exactly")
print(f"{'='*70}")


######################################################################
SAM3 FULL EXTRACTION: ANNOTATIONS + MASKS + fpn FEATURES
######################################################################
Videos:        9
Annotations:   /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Mask coordinates/Teacher_Annotations
Masks:         /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Masks
fpn features: /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/FPN_Features
Local buffer:  /local_disk0/tmp/sam3_extraction
######################################################################


[Video 1/9]: 2019_11_05_000002

Extracting: 2019_11_05_000002
  Frames:       /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Decoded Frames/2019_11_05_000002/video1
  Local temp:   /local_disk0/tmp/sam3_extraction/2019_11_05_000002
  DBFS annot:   /dbfs/mnt/playbehavior/Pig Behavior Edinburgh/Model_Distill/SAM3/Mask coordinates/Teacher_Annotations/2019_11_05_000002
